# Lesson 06: Image Blending — Python (Colab)

This notebook is divided into segments explaining alpha/feather blending and a simple multi-band (Laplacian pyramid) blend for seamless composition.


## 1. Setup and create images with exposure difference

Imports libraries and generates two synthetic overlapping images with a brightness/exposure difference to make seams visible when blended.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_img(offset=0, bright=0):
    H,W=128,256
    y,x=np.mgrid[0:H,0:W]
    base = (((x+offset)//16 + y//16) % 2)*120 + 60
    detail = 20*np.sin((x+offset)*0.3)*np.sin(y*0.3)
    return np.clip(base + detail + bright, 0, 255).astype(np.float32)

A = make_img(0, 0)
B = make_img(0, 35)  # exposure difference


## 2. Alpha/feather blending

Implements weighted blending `I = αA + (1-α)B` with a spatially varying mask. A soft (feathered) transition reduces abrupt seams compared to a hard cut.


In [ ]:
def feather_blend(A,B,center,width):
    H,W=A.shape; alpha=np.ones((H,W),np.float32)
    ramp=np.clip((np.arange(W)- (center-width))/(2*width+1e-6),0,1)
    alpha *= (1-ramp)
    return alpha*A + (1-alpha)*B, alpha


## 3. Gaussian and Laplacian pyramid utilities

Defines downsample/upsample operations and builds Gaussian and Laplacian pyramids. Laplacian levels store band-pass details so an image can be reconstructed across scales.


In [ ]:
# Simple Gaussian/Laplacian pyramid (down/up by factor 2)
def down(img):
    k=np.array([1,4,6,4,1],np.float32)/16
    g=np.apply_along_axis(lambda m: np.convolve(m,k,mode='same'), axis=1, arr=img)
    g=np.apply_along_axis(lambda m: np.convolve(m,k,mode='same'), axis=0, arr=g)
    return g[::2,::2]

def up(img, shape):
    out=np.zeros(shape,np.float32)
    out[::2,::2]=img
    k=np.array([1,4,6,4,1],np.float32)/8
    out=np.apply_along_axis(lambda m: np.convolve(m,k,mode='same'), axis=1, arr=out)
    out=np.apply_along_axis(lambda m: np.convolve(m,k,mode='same'), axis=0, arr=out)
    return out

def gaussian_pyr(img, levels):
    G=[img]
    for _ in range(levels): G.append(down(G[-1]))
    return G

def laplacian_pyr(img, levels):
    G=gaussian_pyr(img,levels); L=[]
    for i in range(levels):
        L.append(G[i]-up(G[i+1], G[i].shape))
    L.append(G[-1]); return L

def blend_pyr(LA,LB,GM):
    return [gm*la + (1-gm)*lb for la,lb,gm in zip(LA,LB,GM)]

def collapse(L):
    img=L[-1]
    for i in range(len(L)-2,-1,-1): img = up(img, L[i].shape) + L[i]
    return img


## 4. Multi-band blending and reconstruction

Blends each Laplacian level using a Gaussian pyramid of the mask, then collapses the pyramid. Low frequencies transition broadly while high frequencies transition narrowly, reducing seams while preserving detail.


In [ ]:
levels=4
center, width = W//2, 20
_, alpha = feather_blend(A,B,center,width)
hard, _ = feather_blend(A,B,center,0)
feath, _ = feather_blend(A,B,center,width)
LA,LB = laplacian_pyr(A,levels), laplacian_pyr(B,levels)
GM = gaussian_pyr(alpha,levels)
pyr = np.clip(collapse(blend_pyr(LA,LB,GM)),0,255)


## 5. Visualize blending results

Compares inputs, mask, hard seam, feather blend, and multi-band pyramid blend side by side.


In [ ]:
fig,axs=plt.subplots(2,3,figsize=(13,6))
axs[0,0].imshow(A,cmap='gray',vmin=0,vmax=255); axs[0,0].set_title('A')
axs[0,1].imshow(B,cmap='gray',vmin=0,vmax=255); axs[0,1].set_title('B (brighter)')
axs[0,2].imshow(alpha,cmap='gray'); axs[0,2].set_title('Mask (feathered)')
axs[1,0].imshow(hard,cmap='gray',vmin=0,vmax=255); axs[1,0].set_title('Hard seam')
axs[1,1].imshow(feath,cmap='gray',vmin=0,vmax=255); axs[1,1].set_title('Feather blend')
axs[1,2].imshow(pyr,cmap='gray',vmin=0,vmax=255); axs[1,2].set_title('Multi-band (pyramid)')
for a in axs.ravel(): a.axis('off')
plt.tight_layout(); plt.show()


## Experiments

- Introduce a few-pixel misalignment between A and B to see ghosting, or blend real images with an alpha mask.
- Vary feather width and pyramid levels to observe trade-offs.
